# SHNY vs SOXL · Bollinger Band & Rolling Correlation (Colab-ready)

- **SHNY** : MicroSectors Gold 3× Leveraged ETN (상장 2023-02-24)
- **SOXL** : Direxion Daily Semiconductor Bull 3× ETF
- 볼린저: 30일 SMA ± 2σ, 상관: 30일 롤링 일간수익률 상관계수

**실제 시세를 yfinance로 받아 사용합니다 (합성 데이터 없음).** Colab에서 셀 단위로 실행하세요.
다운로드한 raw 시세는 `./data/prices.csv`로 캐시되어, 다시 실행하면 네트워크 호출 없이 로드합니다. 강제 재다운로드는 `FORCE_REFRESH = True`로 두면 됩니다.

In [ ]:
# 1. 환경 준비
!pip install -q yfinance

In [ ]:
# 2. import & 파라미터
import os, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter, MonthLocator

TICKERS       = ['SHNY', 'SOXL']
START         = '2023-02-24'   # SHNY 상장일
END           = None           # None이면 오늘까지
WINDOW        = 30             # 볼린저/롤링상관 윈도우
NSTD          = 2              # 볼린저 표준편차 배수
DATA_DIR      = Path('data');  DATA_DIR.mkdir(exist_ok=True)
IMG_DIR       = Path('image'); IMG_DIR.mkdir(exist_ok=True)
CACHE_PATH    = DATA_DIR / 'prices.csv'
FORCE_REFRESH = False          # True로 두면 캐시를 무시하고 재다운로드

In [ ]:
# 3. 데이터 다운로드 (수정종가) — per-ticker fallback + 캐시
#    yf.download(list, ...)는 네트워크 상태/버전에 따라 빈 DataFrame을 반환하는 경우가 있어
#    티커별로 yf.Ticker(t).history(...)를 호출하는 견고한 경로를 메인으로 사용합니다.

def fetch_one(ticker: str, start: str, end=None, retries: int = 3) -> pd.Series:
    last_err = None
    for i in range(retries):
        try:
            hist = yf.Ticker(ticker).history(start=start, end=end, auto_adjust=True)
            if hist is None or hist.empty or 'Close' not in hist.columns:
                raise RuntimeError(f'{ticker}: empty response')
            s = hist['Close'].rename(ticker)
            s.index = pd.to_datetime(s.index).tz_localize(None)
            return s
        except Exception as e:
            last_err = e
            time.sleep(2 ** i)
    raise RuntimeError(f'{ticker} download failed after {retries} retries: {last_err}')

def load_prices(tickers, start, end=None) -> pd.DataFrame:
    if CACHE_PATH.exists() and not FORCE_REFRESH:
        df = pd.read_csv(CACHE_PATH, index_col=0, parse_dates=True)
        if set(tickers).issubset(df.columns) and not df.empty:
            print(f'(cache) loaded {CACHE_PATH}  rows={len(df)}')
            return df[list(tickers)].dropna(how='any').sort_index()
    series = [fetch_one(t, start, end) for t in tickers]
    df = pd.concat(series, axis=1).dropna(how='any').sort_index()
    df.to_csv(CACHE_PATH)
    print(f'(fresh) saved {CACHE_PATH}  rows={len(df)}')
    return df

px = load_prices(TICKERS, START, END)
assert not px.empty, '가격 데이터가 비어 있습니다. 네트워크/티커명을 확인하세요.'
assert set(TICKERS).issubset(px.columns)
print(f'Period       : {px.index.min().date()} → {px.index.max().date()}')
print(f'Trading days : {len(px)}')
px.tail()

In [ ]:
# 4. 볼린저 밴드 (vectorized)
def bollinger(s: pd.Series, w: int = WINDOW, k: float = NSTD) -> pd.DataFrame:
    ma = s.rolling(w).mean()
    sd = s.rolling(w).std()  # ddof=1 (표본표준편차)
    return pd.DataFrame({'ma': ma, 'upper': ma + k*sd, 'lower': ma - k*sd})

bb = {t: bollinger(px[t]) for t in TICKERS}
bb['SHNY'].tail()

In [ ]:
# 5. 일간수익률 · 롤링상관 · 요약통계
ret       = px.pct_change()
roll_corr = ret['SHNY'].rolling(WINDOW).corr(ret['SOXL'])
full_corr = ret['SHNY'].corr(ret['SOXL'])

summary = pd.DataFrame({
    'Total Return %' : (px.iloc[-1] / px.iloc[0] - 1) * 100,
    'Annualized Vol %': ret.std() * np.sqrt(252) * 100,
    'Max Drawdown %' : ((px / px.cummax()) - 1).min() * 100,
}).round(2)

print(f'Full-period correlation (daily returns) : {full_corr:+.4f}')
print(f'30D rolling corr  mean: {roll_corr.mean():+.4f}  std: {roll_corr.std():.4f}')
print(f'30D rolling corr  min : {roll_corr.min():+.4f}  max: {roll_corr.max():+.4f}\n')
summary

In [ ]:
# 6. 시각화 — 3-panel: SHNY BB / SOXL BB / 30D rolling correlation
plt.style.use('dark_background')
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True,
                         gridspec_kw={'height_ratios': [3, 3, 2]})

def draw_bb(ax, ticker, color, dark):
    s, b = px[ticker], bb[ticker]
    ax.plot(s.index, s, color=color, lw=1.4, label=f'{ticker} Close')
    ax.plot(b.index, b['ma'],    color='white', lw=0.9, ls='--', label=f'{WINDOW}D SMA')
    ax.plot(b.index, b['upper'], color=dark,    lw=0.7)
    ax.plot(b.index, b['lower'], color=dark,    lw=0.7)
    ax.fill_between(b.index, b['lower'], b['upper'], color=color, alpha=0.10,
                    label=f'±{NSTD}σ Band')
    ax.set_ylabel(f'{ticker} ($)')
    ax.legend(loc='upper left', fontsize=9, framealpha=0.3)
    ax.grid(alpha=0.2)

draw_bb(axes[0], 'SHNY', '#d4af37', '#806410')
axes[0].set_title('SHNY · MicroSectors Gold 3× Leveraged ETN', loc='left', fontsize=11)
draw_bb(axes[1], 'SOXL', '#3b82f6', '#1e3a8a')
axes[1].set_title('SOXL · Direxion Daily Semiconductor Bull 3× ETF', loc='left', fontsize=11)

ax = axes[2]
ax.plot(roll_corr.index, roll_corr, color='#ef4444', lw=1.3, label=f'{WINDOW}D Rolling ρ')
ax.axhline(0, color='#9ca3af', lw=0.7, ls='--')
ax.axhline(full_corr, color='#10b981', lw=0.9, ls=':',
           label=f'Full ρ = {full_corr:+.3f}')
ax.fill_between(roll_corr.index, 0, roll_corr, where=(roll_corr >= 0),
                color='#10b981', alpha=0.15, interpolate=True)
ax.fill_between(roll_corr.index, 0, roll_corr, where=(roll_corr <  0),
                color='#ef4444', alpha=0.15, interpolate=True)
ax.set_ylim(-1, 1); ax.set_ylabel('Correlation')
ax.set_title('30-Day Rolling Correlation · Daily Returns', loc='left', fontsize=11)
ax.legend(loc='upper left', fontsize=9, framealpha=0.3)
ax.grid(alpha=0.2)
ax.xaxis.set_major_locator(MonthLocator(interval=3))
ax.xaxis.set_major_formatter(DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
out_path = IMG_DIR / 'SHNY_SOXL_bollinger_corr.png'
plt.savefig(out_path, dpi=120, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

In [ ]:
# 7. 보너스 — 일간수익률 산점도 + OLS 회귀선 (β = SOXL을 SHNY에 회귀한 slope)
r = ret.dropna()
slope, intercept = np.polyfit(r['SHNY'], r['SOXL'], 1)
xs = np.linspace(r['SHNY'].min(), r['SHNY'].max(), 100)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(r['SHNY']*100, r['SOXL']*100, alpha=0.45, s=18,
           color='#d4af37', edgecolor='none')
ax.plot(xs*100, (slope*xs + intercept)*100, color='#ef4444', lw=1.6,
        label=f'OLS  β = {slope:+.3f}   ρ = {full_corr:+.3f}')
ax.axhline(0, color='white', lw=0.5, alpha=0.3)
ax.axvline(0, color='white', lw=0.5, alpha=0.3)
ax.set_xlabel('SHNY daily return (%)')
ax.set_ylabel('SOXL daily return (%)')
ax.set_title('Daily Returns Scatter · SHNY vs SOXL', loc='left')
ax.legend(loc='lower right')
ax.grid(alpha=0.2)
plt.tight_layout()
out_path = IMG_DIR / 'SHNY_SOXL_scatter.png'
plt.savefig(out_path, dpi=120, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

### 해석 노트
- 두 자산 모두 3× 데일리 리밸런싱이라 **변동성 드래그**가 누적되어 단순 페어 트레이딩은 권장하기 어렵습니다.
- 일간 수익률 상관은 보통 0 근방에서 진동합니다 — 두 ETF가 추종하는 기초자산(금 vs 반도체 지수)이 서로 다른 매크로 팩터를 따르기 때문입니다.
- 위험회피 국면에서는 일시적으로 음의 상관, 정상장에서는 약한 양의 상관으로 전환되는 패턴이 자주 보입니다 — 30D 롤링 상관 차트의 빨간 음영(ρ < 0) 구간을 확인해 보세요.
- 상관계수는 선형 변환에 대해 불변이므로 ρ(SHNY, SOXL) ≈ ρ(GLD, SOXX) — 더 긴 시계열로 같은 거동을 보고 싶다면 `TICKERS = ['GLD', 'SOXX']`로 바꾸고 `START = '2010-01-01'` 등으로 늘려 실행하면 됩니다.